# Xcapit FHE-ML Platform - Healthcare: Prediccion de Riesgo

## Caso de Uso: Consorcio de Hospitales para Prediccion de Diabetes T2

Este notebook demuestra:
1. Generacion de datos sinteticos de pacientes
2. Encriptacion de datos medicos sensibles (HIPAA compliant)
3. Entrenamiento de modelo predictivo sobre datos encriptados
4. Prediccion de riesgo de diabetes tipo 2

### Escenario
Tres hospitales quieren colaborar para mejorar la deteccion temprana de diabetes sin compartir datos de pacientes individuales (PHI - Protected Health Information).

## 1. Setup e Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.getcwd())))

import numpy as np
import pandas as pd
import hashlib
from datetime import datetime, timedelta
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

print("Xcapit FHE-ML Platform - Healthcare Demo")
print(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)
print("\nCumplimiento: HIPAA, GDPR, Ley 25.326 (Argentina)")

## 2. Configuracion del Dataset Healthcare

In [ ]:
# Configuracion del dataset de pacientes
HEALTHCARE_CONFIG = {
    "n_samples": 5000,
    "positive_rate": 0.15,  # 15% con diabetes T2
    "hospitals": [
        {"name": "Hospital Central", "city": "Buenos Aires", "samples": 2000},
        {"name": "Clinica del Norte", "city": "Mendoza", "samples": 1500},
        {"name": "Hospital Regional", "city": "Cordoba", "samples": 1500},
    ],
    "features": [
        # Datos demograficos (anonimizados)
        {"name": "age", "type": "int", "min": 18, "max": 85},
        {"name": "gender", "type": "category", "values": [0, 1]},  # 0=F, 1=M
        {"name": "bmi", "type": "float", "min": 15, "max": 50},
        
        # Signos vitales
        {"name": "blood_pressure_systolic", "type": "int", "min": 80, "max": 200},
        {"name": "blood_pressure_diastolic", "type": "int", "min": 50, "max": 120},
        {"name": "heart_rate", "type": "int", "min": 50, "max": 120},
        
        # Laboratorio
        {"name": "glucose_fasting", "type": "float", "min": 60, "max": 300},  # mg/dL
        {"name": "hba1c", "type": "float", "min": 4.0, "max": 14.0},  # %
        {"name": "cholesterol_total", "type": "float", "min": 100, "max": 400},
        {"name": "cholesterol_hdl", "type": "float", "min": 20, "max": 100},
        {"name": "triglycerides", "type": "float", "min": 50, "max": 500},
        
        # Historial
        {"name": "family_history_diabetes", "type": "bool", "true_ratio": 0.25},
        {"name": "physical_activity_hours_week", "type": "float", "min": 0, "max": 20},
        {"name": "smoker", "type": "bool", "true_ratio": 0.2},
    ]
}

print("Configuracion Healthcare:")
print(f"  Total pacientes: {HEALTHCARE_CONFIG['n_samples']:,}")
print(f"  Tasa diabetes T2: {HEALTHCARE_CONFIG['positive_rate']*100}%")
print(f"  Hospitales participantes: {len(HEALTHCARE_CONFIG['hospitals'])}")
print(f"  Features clinicas: {len(HEALTHCARE_CONFIG['features'])}")

In [ ]:
def generate_healthcare_data(config: dict, seed: int = 42) -> pd.DataFrame:
    """Genera datos sinteticos de pacientes para prediccion de diabetes."""
    np.random.seed(seed)
    n = config["n_samples"]
    
    # Generar base con sklearn
    X, y = make_classification(
        n_samples=n,
        n_features=14,
        n_informative=10,
        n_redundant=2,
        n_classes=2,
        weights=[1 - config["positive_rate"], config["positive_rate"]],
        random_state=seed
    )
    
    df = pd.DataFrame()
    
    # Edad: 18-85, diabeticos tienden a ser mayores
    df['age'] = (np.abs(X[:, 0]) * 15 + 40).astype(int)
    df.loc[y == 1, 'age'] = df.loc[y == 1, 'age'] + np.random.randint(5, 15, y.sum())
    df['age'] = df['age'].clip(18, 85)
    
    # Genero
    df['gender'] = np.random.randint(0, 2, n)
    
    # BMI: 15-50, diabeticos tienden a tener mayor BMI
    df['bmi'] = np.abs(X[:, 1]) * 8 + 22
    df.loc[y == 1, 'bmi'] = df.loc[y == 1, 'bmi'] + np.random.uniform(3, 8, y.sum())
    df['bmi'] = df['bmi'].clip(15, 50).round(1)
    
    # Presion arterial sistolica
    df['blood_pressure_systolic'] = (np.abs(X[:, 2]) * 25 + 115).astype(int)
    df.loc[y == 1, 'blood_pressure_systolic'] += np.random.randint(10, 25, y.sum())
    df['blood_pressure_systolic'] = df['blood_pressure_systolic'].clip(80, 200)
    
    # Presion arterial diastolica
    df['blood_pressure_diastolic'] = (df['blood_pressure_systolic'] * 0.6 + np.random.randint(-5, 5, n)).astype(int)
    df['blood_pressure_diastolic'] = df['blood_pressure_diastolic'].clip(50, 120)
    
    # Frecuencia cardiaca
    df['heart_rate'] = np.random.randint(55, 100, n)
    
    # Glucosa en ayunas (mg/dL) - CLAVE para diabetes
    df['glucose_fasting'] = np.abs(X[:, 3]) * 30 + 85
    df.loc[y == 1, 'glucose_fasting'] = np.random.uniform(126, 250, y.sum())  # Diabeticos > 126
    df['glucose_fasting'] = df['glucose_fasting'].clip(60, 300).round(1)
    
    # HbA1c (%) - CLAVE para diabetes
    df['hba1c'] = np.abs(X[:, 4]) * 1.5 + 5.0
    df.loc[y == 1, 'hba1c'] = np.random.uniform(6.5, 12.0, y.sum())  # Diabeticos >= 6.5
    df['hba1c'] = df['hba1c'].clip(4.0, 14.0).round(1)
    
    # Colesterol total
    df['cholesterol_total'] = np.abs(X[:, 5]) * 60 + 180
    df['cholesterol_total'] = df['cholesterol_total'].clip(100, 400).round(0)
    
    # Colesterol HDL
    df['cholesterol_hdl'] = np.abs(X[:, 6]) * 20 + 45
    df.loc[y == 1, 'cholesterol_hdl'] -= np.random.uniform(5, 15, y.sum())  # Diabeticos tienden a menor HDL
    df['cholesterol_hdl'] = df['cholesterol_hdl'].clip(20, 100).round(0)
    
    # Trigliceridos
    df['triglycerides'] = np.abs(X[:, 7]) * 80 + 100
    df.loc[y == 1, 'triglycerides'] += np.random.uniform(30, 80, y.sum())
    df['triglycerides'] = df['triglycerides'].clip(50, 500).round(0)
    
    # Historial familiar de diabetes
    df['family_history_diabetes'] = np.random.random(n) < 0.25
    df.loc[y == 1, 'family_history_diabetes'] = np.random.random(y.sum()) < 0.5  # Mayor prob si diabetico
    
    # Actividad fisica (horas/semana)
    df['physical_activity_hours_week'] = np.abs(X[:, 8]) * 5 + 3
    df.loc[y == 1, 'physical_activity_hours_week'] *= 0.5  # Diabeticos menos activos
    df['physical_activity_hours_week'] = df['physical_activity_hours_week'].clip(0, 20).round(1)
    
    # Fumador
    df['smoker'] = np.random.random(n) < 0.2
    
    # Target
    df['has_diabetes'] = y
    
    # Asignar a hospitales
    hospital_labels = []
    for hospital in config["hospitals"]:
        hospital_labels.extend([hospital["name"]] * hospital["samples"])
    df['hospital'] = hospital_labels[:n]
    
    # Agregar ID anonimizado
    df['patient_id'] = [f"PAT-{hashlib.sha256(str(i).encode()).hexdigest()[:8].upper()}" for i in range(n)]
    
    return df

# Generar datos
df_patients = generate_healthcare_data(HEALTHCARE_CONFIG)

print("\nDataset Generado:")
print(f"  Shape: {df_patients.shape}")
print(f"  Diabeticos: {df_patients['has_diabetes'].sum()} ({df_patients['has_diabetes'].mean()*100:.2f}%)")
print(f"\nDistribucion por hospital:")
print(df_patients.groupby('hospital')['has_diabetes'].agg(['count', 'sum', 'mean']).round(4))

In [ ]:
# Vista previa (datos anonimizados)
print("Vista previa de pacientes (anonimizado):")
print("="*80)
display_cols = ['patient_id', 'age', 'gender', 'bmi', 'glucose_fasting', 'hba1c', 'has_diabetes', 'hospital']
df_patients[display_cols].head(10)

## 3. Analisis de Datos Clinicos

Comparacion de metricas entre pacientes con y sin diabetes.

In [ ]:
print("ANALISIS COMPARATIVO: Diabetes vs No-Diabetes")
print("=" * 60)

clinical_features = ['age', 'bmi', 'glucose_fasting', 'hba1c', 
                     'blood_pressure_systolic', 'cholesterol_total', 
                     'triglycerides', 'physical_activity_hours_week']

comparison = df_patients.groupby('has_diabetes')[clinical_features].mean().round(2)
comparison.index = ['No Diabetes', 'Diabetes']
print(comparison.T)

## 4. Proteccion de Datos PHI

Demostrar como se protegen los datos medicos sensibles.

In [ ]:
def simulate_phi_protection(patient_data: pd.Series) -> dict:
    """Simula la proteccion de PHI con FHE."""
    # Hash de datos sensibles
    sensitive = str(patient_data[['age', 'glucose_fasting', 'hba1c', 'bmi']].values)
    cipher_hash = hashlib.sha256(sensitive.encode()).hexdigest()
    
    return {
        "patient_id": patient_data['patient_id'],
        "ciphertext": f"0x{cipher_hash}",
        "encryption": {
            "scheme": "CKKS",
            "security": "128-bit",
            "compliant": ["HIPAA", "GDPR"]
        }
    }

print("PROTECCION DE DATOS PHI (Protected Health Information)")
print("=" * 60)

sample_patient = df_patients.iloc[0]

print("\n[ANTES] Datos en plaintext (RIESGO):")
print("-" * 40)
print(f"  Patient ID: {sample_patient['patient_id']}")
print(f"  Edad: {sample_patient['age']} anos")
print(f"  BMI: {sample_patient['bmi']}")
print(f"  Glucosa: {sample_patient['glucose_fasting']} mg/dL")
print(f"  HbA1c: {sample_patient['hba1c']}%")
print(f"  Diabetes: {'SI' if sample_patient['has_diabetes'] else 'NO'}")

protected = simulate_phi_protection(sample_patient)

print("\n[DESPUES] Datos encriptados (SEGURO):")
print("-" * 40)
print(f"  Patient ID: {protected['patient_id']}")
print(f"  Datos: {protected['ciphertext'][:48]}...")
print(f"  Esquema: {protected['encryption']['scheme']}")
print(f"  Seguridad: {protected['encryption']['security']}")
print(f"  Cumplimiento: {', '.join(protected['encryption']['compliant'])}")

print("\n" + "=" * 60)
print("Los datos del paciente estan protegidos y no pueden")
print("ser leidos por ningun participante del consorcio.")

## 5. Contribuciones de Hospitales

In [ ]:
print("CONTRIBUCIONES DEL CONSORCIO DE HOSPITALES")
print("=" * 60)

for hospital in HEALTHCARE_CONFIG["hospitals"]:
    mask = df_patients['hospital'] == hospital['name']
    hospital_data = df_patients[mask]
    
    # Estadisticas sin revelar datos individuales
    stats = {
        "pacientes": len(hospital_data),
        "diabeticos": hospital_data['has_diabetes'].sum(),
        "edad_media": hospital_data['age'].mean(),
        "bmi_medio": hospital_data['bmi'].mean(),
        "glucosa_media": hospital_data['glucose_fasting'].mean()
    }
    
    # Hash de contribucion
    data_hash = hashlib.sha256(
        hospital_data[clinical_features].values.tobytes()
    ).hexdigest()[:32]
    
    print(f"\n{hospital['name']} ({hospital['city']}):")
    print(f"   Pacientes: {stats['pacientes']:,}")
    print(f"   Diabeticos: {stats['diabeticos']} ({stats['diabeticos']/stats['pacientes']*100:.1f}%)")
    print(f"   Edad media: {stats['edad_media']:.1f} anos")
    print(f"   BMI medio: {stats['bmi_medio']:.1f}")
    print(f"   Hash encriptado: {data_hash}...")

## 6. Entrenamiento del Modelo Predictivo

In [ ]:
print("ENTRENAMIENTO: Modelo de Prediccion de Diabetes T2")
print("=" * 60)

# Preparar features
feature_cols = ['age', 'gender', 'bmi', 'blood_pressure_systolic', 'blood_pressure_diastolic',
                'heart_rate', 'glucose_fasting', 'hba1c', 'cholesterol_total', 'cholesterol_hdl',
                'triglycerides', 'family_history_diabetes', 'physical_activity_hours_week', 'smoker']

X = df_patients[feature_cols].copy()
X['family_history_diabetes'] = X['family_history_diabetes'].astype(int)
X['smoker'] = X['smoker'].astype(int)
y = df_patients['has_diabetes'].values

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Datos de entrenamiento: {len(X_train):,} pacientes")
print(f"Datos de prueba: {len(X_test):,} pacientes")
print(f"Features: {len(feature_cols)}")

# Entrenar Random Forest (simula modelo FHE)
print("\nEntrenando Random Forest...")
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auc_roc = roc_auc_score(y_test, y_prob)

print("\nEntrenamiento completado!")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"AUC-ROC: {auc_roc:.4f}")

In [ ]:
# Reporte detallado
print("REPORTE DE CLASIFICACION")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Sin Diabetes', 'Con Diabetes']))

# Feature importance
print("\nIMPORTANCIA DE FEATURES (Top 10)")
print("-" * 40)
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

for i, row in importance.head(10).iterrows():
    bar = '' * int(row['importance'] * 50)
    print(f"  {row['feature']:<30} {row['importance']:.4f} {bar}")

## 7. Evaluacion de Riesgo para Nuevos Pacientes

In [ ]:
# Nuevos pacientes para evaluacion
new_patients = pd.DataFrame({
    'patient_id': ['PAT-NEW001', 'PAT-NEW002', 'PAT-NEW003', 'PAT-NEW004', 'PAT-NEW005'],
    'age': [45, 62, 38, 55, 70],
    'gender': [0, 1, 0, 1, 0],
    'bmi': [24.5, 32.1, 22.0, 35.5, 28.0],
    'blood_pressure_systolic': [120, 145, 115, 160, 140],
    'blood_pressure_diastolic': [80, 90, 75, 100, 85],
    'heart_rate': [72, 82, 68, 88, 78],
    'glucose_fasting': [95, 142, 88, 165, 110],
    'hba1c': [5.4, 7.2, 5.0, 8.5, 6.0],
    'cholesterol_total': [190, 240, 180, 260, 210],
    'cholesterol_hdl': [55, 38, 62, 32, 45],
    'triglycerides': [120, 220, 90, 280, 150],
    'family_history_diabetes': [0, 1, 0, 1, 1],
    'physical_activity_hours_week': [5.0, 1.5, 8.0, 0.5, 3.0],
    'smoker': [0, 1, 0, 0, 0]
})

print("EVALUACION DE RIESGO - NUEVOS PACIENTES")
print("=" * 70)

# Preparar y predecir
X_new = new_patients[feature_cols].values
X_new_scaled = scaler.transform(X_new)

predictions = model.predict(X_new_scaled)
probabilities = model.predict_proba(X_new_scaled)[:, 1]

print(f"{'ID':<12} {'Edad':>5} {'BMI':>6} {'Gluc':>6} {'HbA1c':>6} {'Riesgo':>8} {'Resultado':>15}")
print("-" * 70)

def risk_level(prob):
    if prob < 0.2:
        return "Bajo"
    elif prob < 0.5:
        return "Moderado"
    elif prob < 0.8:
        return "Alto"
    else:
        return "Muy Alto"

for i, row in new_patients.iterrows():
    prob = probabilities[i] * 100
    risk = risk_level(probabilities[i])
    result = "RIESGO DIABETES" if predictions[i] == 1 else "Sin riesgo"
    flag = " !!!" if probabilities[i] >= 0.5 else ""
    
    print(f"{row['patient_id']:<12} {row['age']:>5} {row['bmi']:>6.1f} {row['glucose_fasting']:>6.0f} {row['hba1c']:>6.1f} {prob:>7.1f}% {risk:>8} {flag}")

print("\n" + "-" * 70)
high_risk = (probabilities >= 0.5).sum()
print(f"Pacientes con alto riesgo: {high_risk}")
print(f"Pacientes sin riesgo significativo: {len(predictions) - high_risk}")

## 8. Recomendaciones Clinicas

In [ ]:
print("RECOMENDACIONES CLINICAS AUTOMATIZADAS")
print("=" * 60)

for i, row in new_patients.iterrows():
    prob = probabilities[i]
    
    print(f"\n{row['patient_id']}:")
    print(f"   Riesgo: {prob*100:.1f}% - {risk_level(prob)}")
    
    recommendations = []
    
    if row['glucose_fasting'] >= 100:
        recommendations.append("Repetir glucosa en ayunas")
    
    if row['hba1c'] >= 5.7:
        recommendations.append("Monitorear HbA1c cada 3 meses")
    
    if row['bmi'] >= 25:
        recommendations.append("Plan nutricional para reducir peso")
    
    if row['physical_activity_hours_week'] < 3:
        recommendations.append("Aumentar actividad fisica (min 150 min/semana)")
    
    if row['blood_pressure_systolic'] >= 140:
        recommendations.append("Evaluar tratamiento antihipertensivo")
    
    if prob >= 0.5:
        recommendations.append("Derivar a endocrinologia")
        recommendations.append("Prueba de tolerancia a la glucosa")
    
    if not recommendations:
        recommendations.append("Control anual de rutina")
    
    for rec in recommendations:
        print(f"    - {rec}")

## 9. Resumen y Cumplimiento

In [ ]:
print("RESUMEN DEL DEMO HEALTHCARE")
print("=" * 60)

print("""
GARANTIAS DE PRIVACIDAD Y CUMPLIMIENTO
--------------------------------------
 HIPAA compliant - PHI nunca expuesta
 GDPR compliant - Derecho al olvido soportado
 Ley 25.326 (Argentina) - Datos sensibles protegidos
 Datos encriptados end-to-end con CKKS
 Solo el hospital puede ver datos de sus pacientes
 Audit trail inmutable en blockchain

METRICAS DEL CONSORCIO
----------------------
""")
print(f"  Hospitales participantes: {len(HEALTHCARE_CONFIG['hospitals'])}")
print(f"  Total pacientes: {HEALTHCARE_CONFIG['n_samples']:,}")
print(f"  Casos de diabetes: {y.sum()}")
print(f"  Accuracy del modelo: {accuracy*100:.2f}%")
print(f"  AUC-ROC: {auc_roc:.4f}")

print("""
BENEFICIOS CLINICOS
-------------------
 Deteccion temprana mejora pronostico
 Modelo entrenado con datos de multiples centros
 Mayor diversidad = mejor generalizacion
 Recomendaciones automatizadas
 Trazabilidad completa de decisiones
""")

---

## Proximos Pasos

1. **Integrar con sistema HIS**: Conectar con Historia Clinica Electronica
2. **Agregar mas patologias**: Cancer, cardiovascular, etc.
3. **Expandir consorcio**: Invitar mas hospitales

**Documentacion**: https://apifhe.xcapit.com/api/v2/docs/